<a href="https://colab.research.google.com/github/softdataconsult/ML-Project/blob/main/company_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import random
from collections import defaultdict
from openpyxl import load_workbook, Workbook

# ============================================================
# 1. HELPERS
# ============================================================

def clean_team(team):
    valid = ["A", "B", "C", "D", "E", "F", "G", "H"]
    if team in valid:
        return team
    return "H"  # fallback to lowest priority

# ============================================================
# 2. LOAD STAFF + DESK DATA FROM EXCEL
# ============================================================

input_file = "/template_project.xlsx"
wb_in = load_workbook(input_file, data_only=True)

sheet1 = wb_in["Sheet1"]
sheet2 = wb_in["Sheet2"]

# ---- Load staff + team from Sheet1 ----
staff_list = []
for row in sheet1.iter_rows(min_row=7, values_only=True):
    name = row[0]
    team = row[2]
    if name:
        staff_list.append((name, clean_team(team)))

# ---- Load desk request + desk type + desk ID from Sheet2 ----
desk_info = {}
desk_inventory = []  # list of (desk_id, desk_type)

for row in sheet2.iter_rows(min_row=2, values_only=True):
    name = row[0]
    desk_request = row[1]
    team = row[2]
    desk_type = row[3]
    desk_id = row[4]

    if name:
        desk_info[name] = (desk_request, desk_type, desk_id)

    if desk_id:
        desk_inventory.append((desk_id, desk_type))

# Keep desk order EXACTLY as Sheet2
desk_ids_in_order = [d[0] for d in desk_inventory]

# Merge staff + desk info
staff_data = []
for name, team in staff_list:
    req, dtype, did = desk_info.get(name, ("Regular", None, None))
    staff_data.append((name, team, req, dtype, did))

names = [s[0] for s in staff_data]

# ============================================================
# 3. CONSTANTS
# ============================================================

weeks = ["1", "2", "3", "4"]
days = ["MON", "TUES", "WED", "THUR", "FRI", "SAT", "SUN"]
weekday_only = days[:5]

# MON1..SUN4
columns = [f"{d}{w}" for w in weeks for d in days]

# Team priority
team_priority = ["A", "B", "C", "D", "E", "F", "G", "H"]

# Desk type lookup
desk_type_map = {desk_id: desk_type for desk_id, desk_type in desk_inventory}

# ============================================================
# 4. SELECT PART-TIME STAFF
# ============================================================

pt_count = random.randint(3, 10)
part_time_staff = set(random.sample(names, pt_count))

# ============================================================
# 5. FIXED NOT-WORKING DAYS
# ============================================================

def choose_not_working(is_pt):
    if is_pt:
        return set(random.sample(weekday_only, 2))
    else:
        k = random.randint(0, 1)
        return set(random.sample(weekday_only, k)) if k else set()

not_working_days = {
    name: choose_not_working(name in part_time_staff)
    for name in names
}

# ============================================================
# 6. INITIAL SCHEDULE
# ============================================================

def random_status():
    return random.choice(["Office", "Home", "Leave"])

schedule = defaultdict(dict)

for name, team, req, dtype, did in staff_data:
    nw = not_working_days[name]
    for w in weeks:
        for d in days:
            col = f"{d}{w}"
            if d in ["SAT", "SUN"]:
                schedule[name][col] = "None"
            elif d in nw:
                schedule[name][col] = "Not working"
            else:
                schedule[name][col] = random_status()

# ============================================================
# 7. ENFORCE ≥60% OFFICE
# ============================================================

def enforce_minimum_office():
    for name, team, req, dtype, did in staff_data:
        is_pt = name in part_time_staff
        target = 7 if is_pt else 12

        office_days = [c for c in columns if schedule[name][c] == "Office"]
        other_days = [c for c in columns if schedule[name][c] in ["Home", "Leave"]]

        if len(office_days) >= target:
            continue

        random.shuffle(other_days)
        idx = 0
        while len(office_days) < target and idx < len(other_days):
            col = other_days[idx]
            schedule[name][col] = "Office"
            office_days.append(col)
            idx += 1

enforce_minimum_office()

# ============================================================
# 8. DESK ALLOCATION (TEAM PRIORITY + EQUAL FALLBACK)
# ============================================================

desk_alloc = {desk_id: {col: "None" for col in columns} for desk_id in desk_ids_in_order}
oversub = []

def allocate_day(col):
    available_desks = set(desk_ids_in_order)

    # sort staff by team priority safely
    def team_rank(team):
        return team_priority.index(team) if team in team_priority else len(team_priority)

    sorted_staff = sorted(staff_data, key=lambda x: team_rank(x[1]))

    for name, team, req, dtype, did in sorted_staff:
        if schedule[name][col] != "Office":
            continue

        preferred = []
        fallback = []

        for desk_id in available_desks:
            d_type = desk_type_map[desk_id] or ""
            if dtype and dtype in d_type:
                preferred.append(desk_id)
            else:
                fallback.append(desk_id)

        chosen = None
        if preferred:
            chosen = random.choice(preferred)
        elif fallback:
            chosen = random.choice(fallback)

        if chosen:
            desk_alloc[chosen][col] = name
            available_desks.remove(chosen)
        else:
            oversub.append((col, name, team, req, "NO DESK AVAILABLE"))

for col in columns:
    allocate_day(col)

# ============================================================
# 9. WRITE OUTPUT EXCEL
# ============================================================

wb_out = Workbook()

# Sheet1: Desk-centric
ws1 = wb_out.active
ws1.title = "Desk_Centric"
ws1.append(["Desk_ID"] + columns)

for desk_id in desk_ids_in_order:
    row = [desk_id] + [desk_alloc[desk_id][col] for col in columns]
    ws1.append(row)

# Sheet2: Oversubscription
ws2 = wb_out.create_sheet("OverSubscribed")
ws2.append(["Column", "Staff", "Team", "Request", "Reason"])
for row in oversub:
    ws2.append(list(row))

# Sheet3: Staff-centric
ws3 = wb_out.create_sheet("Staff_Centric")
ws3.append(["Staff", "Team", "Contract"] + columns)

for name, team, req, dtype, did in staff_data:
    contract = "PT" if name in part_time_staff else "FT"
    row = [name, team, contract] + [schedule[name][col] for col in columns]
    ws3.append(row)

wb_out.save("generated_schedule_desk_centric.xlsx")
print("generated_schedule_desk_centric.xlsx created successfully.")


generated_schedule_desk_centric.xlsx created successfully.


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
